# 19. Web Scraping Agent with Search API Integration
**Industry:** Retail & E-Commerce

Build an agent that searches the web using the Serper API, scrapes the top results with BeautifulSoup, and summarizes findings.

In [ ]:
!pip install langchain langchain-google-genai requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
import os

# You need a Serper API key from https://serper.dev/
SERPER_API_KEY = os.environ.get("SERPER_API_KEY", "YOUR_SERPER_API_KEY_HERE")

def search_google(query: str):
    if SERPER_API_KEY == "YOUR_SERPER_API_KEY_HERE":
        return ["https://example.com/mock-pricing"]
    
    url = "https://google.serper.dev/search"
    payload = {"q": query, "num": 3}
    headers = {
        'X-API-KEY': SERPER_API_KEY,
        'Content-Type': 'application/json'
    }
    response = requests.post(url, headers=headers, json=payload)
    data = response.json()
    return [item.get('link') for item in data.get('organic', [])]

def scrape_url(url: str):
    if "mock-pricing" in url:
        return "Competitor A is selling running shoes at $50 (20% discount)."
    
    try:
        response = requests.get(url, timeout=5)
        soup = BeautifulSoup(response.text, 'html.parser')
        # Extract paragraphs
        paragraphs = soup.find_all('p')
        text = " ".join([p.get_text() for p in paragraphs])
        return text[:1000] # Limit to 1000 chars per page
    except:
        return ""

def research_query(query: str):
    urls = search_google(query)
    combined_content = ""
    for u in urls:
        print(f"Scraping {u}...")
        combined_content += scrape_url(u) + "\n\n"
    
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
    prompt = ChatPromptTemplate.from_template("Summarize this pricing data from competitors:\n\n{data}")
    chain = prompt | llm
    
    summary = chain.invoke({"data": combined_content})
    return summary.content

print(research_query("latest discount trends for nike running shoes"))